# CNN Transfer Learning Pipeline

This notebook trains a CNN pairwise classifier on `han_pairwise_dataset.csv` using an 8:1:1 Train/Development/Test split. The Development set is used for early stopping, and the Test set is used for Han test evaluation. The trained CNN model and tokenizer are saved for later Transfer Learning evaluation on `pairwise_gold_dataset.csv`.

## 1. Install and import

In [1]:
!pip install pandas scikit-learn tensorflow

  Using cached scikit_learn-1.7.2-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (11 kB)
  Using cached scipy-1.15.3-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
  Using cached flatbuffers-25.12.19-py2.py3-none-any.whl.metadata (1.0 kB)
  Using cached ml_dtypes-0.6.0-cp310-cp310-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (8.8 kB)
Using cached scikit_learn-1.7.2-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (9.7 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.2/572.2 MB 61.9 MB/s  0:00:05m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 246.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 217.7 MB/s  0:00:00
Using cached flatbuffers-25.12.19-py2.py3-none-any.whl (26 kB)
Using cached joblib-1.5.3-py3-none-any.whl (309 kB)
   ━━━━━━━━━━━

In [2]:
import os
import json
import pickle
import random
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_recall_fscore_support, classification_report, accuracy_score, roc_auc_score
from sklearn.utils.class_weight import compute_class_weight

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import Input, Embedding, Conv1D, GlobalMaxPooling1D, Dense, Dropout, Concatenate
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

# Set random seed for reproducibility
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

print("TensorFlow version:", tf.__version__)
print("Available GPUs:", tf.config.list_physical_devices("GPU"))

I0000 00:00:1787611905.082101    9417 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1787611905.147389    9417 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1787611907.008016    9417 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


TensorFlow version: 2.21.0
Available GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## 2. Configuration

In [3]:
# Input dataset
DATA_PATH = "han_pairwise_dataset.csv"

# Output folders
OUTPUT_DIR = Path("cnn_transfer_outputs")
MODEL_DIR = OUTPUT_DIR / "saved_models"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# CNN input settings: keep these consistent with the In-domain CNN notebook
MAX_WORDS = 20000        # Maximum number of words in the tokenizer vocabulary
MAX_LEN = 64             # Maximum sequence length for each pair text
EMBEDDING_DIM = 128      # Dimension of trainable word embeddings

# Training settings: keep these consistent with the In-domain CNN notebook when comparing results
EPOCHS = 30
BATCH_SIZE = 32
PATIENCE = 3
THRESHOLD = 0.5

# If you want to exactly match the In-domain CNN class weight setting, use this:
CLASS_WEIGHT_MODE = "fixed"  # options: "fixed", "balanced", None
FIXED_CLASS_WEIGHT = {0: 1.0, 1: 5.0}

# Saved artifact paths
MODEL_PATH = MODEL_DIR / "CNN_han_transfer_model.keras"
TOKENIZER_PATH = MODEL_DIR / "CNN_han_transfer_tokenizer.pkl"
CONFIG_PATH = MODEL_DIR / "CNN_han_transfer_config.json"

## 3. Load Han pairwise dataset

In [4]:
df = pd.read_csv(DATA_PATH, encoding="utf-8-sig")

required_cols = ["OrgName_1", "OrgName_2", "Label"]
missing_cols = [col for col in required_cols if col not in df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

df = df[required_cols].dropna().copy().reset_index(drop=True)
df["OrgName_1"] = df["OrgName_1"].astype(str)
df["OrgName_2"] = df["OrgName_2"].astype(str)
df["Label"] = df["Label"].astype(int)

print("Dataset shape:", df.shape)
print("Label distribution:")
print(df["Label"].value_counts().sort_index())
print(df.head())

Dataset shape: (905357, 3)
Label distribution:
Label
0    715157
1    190200
Name: count, dtype: int64
                                           OrgName_1  \
0  CNPqConselho Nacional de Desenvolvimento Cient...   
1  CNPqConselho Nacional de Desenvolvimento Cient...   
2  CNPqConselho Nacional de Desenvolvimento Cient...   
3  CNPqConselho Nacional de Desenvolvimento Cient...   
4  CNPqConselho Nacional de Desenvolvimento Cient...   

                                           OrgName_2  Label  
0                      Conselho Nacional de Pesquisa      1  
1                                               CNPq      1  
2  National Council for Scientific and Technologi...      1  
3                      Conselho Nacional de Pesquisa      1  
4  Conselho Nacional de Desenvolvimento Científic...      1  


## 4. Train/Development/Test split = 8:1:1

In [5]:
# First split: 80% train and 20% temporary set
train_df, temp_df = train_test_split(
    df,
    test_size=0.20,
    stratify=df["Label"],
    random_state=RANDOM_SEED
)

# Second split: split temporary set into 10% development and 10% test
# Since temp_df is 20% of the full dataset, test_size=0.50 makes dev/test each 10% of the full dataset.
dev_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["Label"],
    random_state=RANDOM_SEED
)

train_df = train_df.reset_index(drop=True)
dev_df = dev_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("Train:", train_df.shape, train_df["Label"].value_counts().sort_index().to_dict())
print("Development:", dev_df.shape, dev_df["Label"].value_counts().sort_index().to_dict())
print("Test:", test_df.shape, test_df["Label"].value_counts().sort_index().to_dict())

train_df.to_csv(OUTPUT_DIR / "han_train_80pct.csv", index=False, encoding="utf-8-sig")
dev_df.to_csv(OUTPUT_DIR / "han_development_10pct.csv", index=False, encoding="utf-8-sig")
test_df.to_csv(OUTPUT_DIR / "han_test_10pct.csv", index=False, encoding="utf-8-sig")

Train: (724285, 3) {0: 572125, 1: 152160}
Development: (90536, 3) {0: 71516, 1: 19020}
Test: (90536, 3) {0: 71516, 1: 19020}


## 5. CNN input preparation

In [6]:
def make_pair_text(dataframe):
    # Combine two organization names into one input sequence
    return (
        dataframe["OrgName_1"].astype(str)
        + " [SEP] "
        + dataframe["OrgName_2"].astype(str)
    ).tolist()

train_text = make_pair_text(train_df)
dev_text = make_pair_text(dev_df)
test_text = make_pair_text(test_df)

y_train = train_df["Label"].values
y_dev = dev_df["Label"].values
y_test = test_df["Label"].values

# Fit tokenizer only on training text to avoid data leakage
tokenizer = Tokenizer(
    num_words=MAX_WORDS,
    oov_token="[OOV]"
)
tokenizer.fit_on_texts(train_text)

def texts_to_padded_sequences(texts, tokenizer):
    return pad_sequences(
        tokenizer.texts_to_sequences(texts),
        maxlen=MAX_LEN,
        padding="post",
        truncating="post"
    )

X_train = texts_to_padded_sequences(train_text, tokenizer)
X_dev = texts_to_padded_sequences(dev_text, tokenizer)
X_test = texts_to_padded_sequences(test_text, tokenizer)

vocab_size = min(MAX_WORDS, len(tokenizer.word_index) + 1)

print("X_train:", X_train.shape)
print("X_dev:", X_dev.shape)
print("X_test:", X_test.shape)
print("Effective vocabulary size:", vocab_size)

X_train: (724285, 64)
X_dev: (90536, 64)
X_test: (90536, 64)
Effective vocabulary size: 20000


## 6. CNN model

In [7]:
def build_cnn_model(vocab_size):
    # Define CNN architecture for pairwise text classification
    input_layer = Input(shape=(MAX_LEN,))

    # Convert token IDs into dense trainable embeddings
    x = Embedding(
        input_dim=vocab_size,
        output_dim=EMBEDDING_DIM,
        input_length=MAX_LEN
    )(input_layer)

    # Apply convolution filters with different kernel sizes
    conv3 = Conv1D(filters=128, kernel_size=3, activation="relu")(x)
    conv4 = Conv1D(filters=128, kernel_size=4, activation="relu")(x)
    conv5 = Conv1D(filters=128, kernel_size=5, activation="relu")(x)

    # Apply global max pooling to each convolution output
    pool3 = GlobalMaxPooling1D()(conv3)
    pool4 = GlobalMaxPooling1D()(conv4)
    pool5 = GlobalMaxPooling1D()(conv5)

    # Concatenate pooled features from different kernel sizes
    x = Concatenate()([pool3, pool4, pool5])

    # Add dropout to reduce overfitting
    x = Dropout(0.5)(x)

    # Add dense layer for classification
    x = Dense(128, activation="relu")(x)

    # Add another dropout layer
    x = Dropout(0.3)(x)

    # Output probability of match class
    output_layer = Dense(1, activation="sigmoid")(x)

    # Build model
    model = Model(inputs=input_layer, outputs=output_layer)

    # Compile model for binary classification
    model.compile(
        optimizer="adam",
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    return model

model = build_cnn_model(vocab_size)
model.summary()

/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(
I0000 00:00:1787611948.632451    9417 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:1e.0, compute capability: 7.5


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 64)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 64, 128)   │  2,560,000 │ input_layer[0][0] │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d (Conv1D)     │ (None, 62, 128)   │     49,280 │ embedding[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_1 (Conv1D)   │ (None, 61, 128)   │     65,664 │ embedding[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_2 (Conv1D)   │ (None, 60, 128)   │     82,048 │ embedding[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_max_pooling… │ (None, 128)       │          0 │ conv1d[0][0]      │
│ (GlobalMaxPooling1… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_max_pooling… │ (None, 128)       │          0 │ conv1d_1[0][0]    │
│ (GlobalMaxPooling1… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_max_pooling… │ (None, 128)       │          0 │ conv1d_2[0][0]    │
│ (GlobalMaxPooling1… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 384)       │          0 │ global_max_pooli… │
│ (Concatenate)       │                   │            │ global_max_pooli… │
│                     │                   │            │ global_max_pooli… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 384)       │          0 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 128)       │     49,280 │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 128)       │          0 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 1)         │        129 │ dropout_1[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 2,806,401 (10.71 MB)

 Trainable params: 2,806,401 (10.71 MB)

 Non-trainable params: 0 (0.00 B)

## 7. Set class weights

In [8]:
if CLASS_WEIGHT_MODE == "fixed":
    class_weight = FIXED_CLASS_WEIGHT
elif CLASS_WEIGHT_MODE == "balanced":
    classes = np.array([0, 1])
    weights = compute_class_weight(
        class_weight="balanced",
        classes=classes,
        y=y_train
    )
    class_weight = {int(cls): float(w) for cls, w in zip(classes, weights)}
else:
    class_weight = None

print("Class weight:", class_weight)

Class weight: {0: 1.0, 1: 5.0}


## 8. Train CNN on Han training set

In [9]:
early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=PATIENCE,
    restore_best_weights=True
)

model_checkpoint = ModelCheckpoint(
    filepath=str(MODEL_PATH),
    monitor="val_loss",
    save_best_only=True,
    save_weights_only=False
)

history = model.fit(
    X_train,
    y_train,
    validation_data=(X_dev, y_dev),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[early_stopping, model_checkpoint],
    verbose=1,
    class_weight=class_weight
)

# Load the best validation-loss model saved by ModelCheckpoint
model = load_model(MODEL_PATH)
print("Best CNN model saved to:", MODEL_PATH)

Epoch 1/30


I0000 00:00:1787611952.681801    9901 service.cc:153] XLA service 0x7fef8c033920 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1787611952.681824    9901 service.cc:161]   StreamExecutor [0]: Tesla T4, Compute Capability 7.5 (Driver: 13.2.0; Runtime: 12.8.0; Toolkit: 12.5.0; DNN: 9.10.2)
I0000 00:00:1787611952.724621    9901 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1787611953.047671    9901 cuda_dnn.cc:461] Loaded cuDNN version 91002
I0000 00:00:1787611953.124558    9901 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_2573__.21


   55/22634 ━━━━━━━━━━━━━━━━━━━━ 1:04 3ms/step - accuracy: 0.2863 - loss: 1.2313

I0000 00:00:1787611963.829831    9901 device_compiler.h:208] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


22630/22634 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8424 - loss: 0.6337

I0000 00:00:1787612027.920384    9900 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_2573__.21


22634/22634 ━━━━━━━━━━━━━━━━━━━━ 86s 3ms/step - accuracy: 0.8718 - loss: 0.5680 - val_accuracy: 0.9000 - val_loss: 0.2985
Epoch 2/30
22634/22634 ━━━━━━━━━━━━━━━━━━━━ 69s 3ms/step - accuracy: 0.8969 - loss: 0.4816 - val_accuracy: 0.9091 - val_loss: 0.2832
Epoch 3/30
22634/22634 ━━━━━━━━━━━━━━━━━━━━ 68s 3ms/step - accuracy: 0.9042 - loss: 0.4527 - val_accuracy: 0.9143 - val_loss: 0.2728
Epoch 4/30
22634/22634 ━━━━━━━━━━━━━━━━━━━━ 69s 3ms/step - accuracy: 0.9086 - loss: 0.4350 - val_accuracy: 0.9129 - val_loss: 0.2775
Epoch 5/30
22634/22634 ━━━━━━━━━━━━━━━━━━━━ 69s 3ms/step - accuracy: 0.9119 - loss: 0.4254 - val_accuracy: 0.9188 - val_loss: 0.2591
Epoch 6/30
22634/22634 ━━━━━━━━━━━━━━━━━━━━ 68s 3ms/step - accuracy: 0.9153 - loss: 0.4172 - val_accuracy: 0.9116 - val_loss: 0.2662
Epoch 7/30
22634/22634 ━━━━━━━━━━━━━━━━━━━━ 68s 3ms/step - accuracy: 0.9178 - loss: 0.4095 - val_accuracy: 0.9175 - val_loss: 0.2693
Epoch 8/30
22634/22634 ━━━━━━━━━━━━━━━━━━━━ 68s 3ms/step - accuracy: 0.9193 - lo

## 9. Evaluate CNN on the Han test set

In [10]:
def evaluate_predictions(y_true, y_prob, model_name="CNN", threshold=0.5):
    y_pred = (y_prob >= threshold).astype(int)

    class_precision, class_recall, class_f1, class_support = precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=[0, 1],
        average=None,
        zero_division=0
    )

    summary_df = pd.DataFrame([
        {
            "Model": model_name,
            "Class": "NonMatch",
            "Precision": class_precision[0],
            "Recall": class_recall[0],
            "F1": class_f1[0],
            "Support": class_support[0],
            "Accuracy": accuracy_score(y_true, y_pred),
            "ROC_AUC": roc_auc_score(y_true, y_prob)
        },
        {
            "Model": model_name,
            "Class": "Match",
            "Precision": class_precision[1],
            "Recall": class_recall[1],
            "F1": class_f1[1],
            "Support": class_support[1],
            "Accuracy": accuracy_score(y_true, y_pred),
            "ROC_AUC": roc_auc_score(y_true, y_prob)
        }
    ])

    print(classification_report(
        y_true,
        y_pred,
        target_names=["NonMatch", "Match"],
        digits=4,
        zero_division=0
    ))
    print("Accuracy:", accuracy_score(y_true, y_pred))
    print("ROC_AUC:", roc_auc_score(y_true, y_prob))

    return y_pred, summary_df

# Predict match probability for Han test pairs
y_prob_test = model.predict(X_test, batch_size=BATCH_SIZE).ravel()
y_pred_test, cnn_summary_df = evaluate_predictions(
    y_test,
    y_prob_test,
    model_name="CNN",
    threshold=THRESHOLD
)

# Save pair-level prediction results
cnn_pair_predictions_df = test_df.copy()
cnn_pair_predictions_df["Model"] = "CNN"
cnn_pair_predictions_df["TrueLabel"] = y_test
cnn_pair_predictions_df["PredictedLabel"] = y_pred_test
cnn_pair_predictions_df["Probability_Match"] = y_prob_test
cnn_pair_predictions_df["Probability_NonMatch"] = 1 - y_prob_test

cnn_pair_predictions_df.to_csv(
    OUTPUT_DIR / "cnn_pair_level_predictions_with_probability.csv",
    index=False,
    encoding="utf-8-sig"
)

cnn_summary_df.to_csv(
    OUTPUT_DIR / "cnn_summary_match_nonmatch.csv",
    index=False,
    encoding="utf-8-sig"
)

print(cnn_summary_df)

2830/2830 ━━━━━━━━━━━━━━━━━━━━ 3s 1ms/step
              precision    recall  f1-score   support

    NonMatch     0.9522    0.9456    0.9489     71516
       Match     0.8007    0.8215    0.8109     19020

    accuracy                         0.9195     90536
   macro avg     0.8764    0.8835    0.8799     90536
weighted avg     0.9204    0.9195    0.9199     90536

Accuracy: 0.9195347706989485
ROC_AUC: 0.9552702754184293
  Model     Class  Precision    Recall        F1  Support  Accuracy  ROC_AUC
0   CNN  NonMatch   0.952185  0.945621  0.948891    71516  0.919535  0.95527
1   CNN     Match   0.800697  0.821451  0.810941    19020  0.919535  0.95527


## 10. Save tokenizer and configuration

In [11]:
with open(TOKENIZER_PATH, "wb") as f:
    pickle.dump(tokenizer, f)

config = {
    "model_name": "CNN",
    "max_words": MAX_WORDS,
    "max_len": MAX_LEN,
    "embedding_dim": EMBEDDING_DIM,
    "batch_size": BATCH_SIZE,
    "threshold": THRESHOLD,
    "class_weight_mode": CLASS_WEIGHT_MODE,
    "fixed_class_weight": FIXED_CLASS_WEIGHT,
    "vocab_size": int(vocab_size),
    "random_seed": RANDOM_SEED
}

with open(CONFIG_PATH, "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2)

print("Saved CNN model:", MODEL_PATH)
print("Saved tokenizer:", TOKENIZER_PATH)
print("Saved config:", CONFIG_PATH)

Saved CNN model: cnn_transfer_outputs/saved_models/CNN_han_transfer_model.keras
Saved tokenizer: cnn_transfer_outputs/saved_models/CNN_han_transfer_tokenizer.pkl
Saved config: cnn_transfer_outputs/saved_models/CNN_han_transfer_config.json


## 11. Function for applying the saved CNN model to pairwise_gold_dataset.csv

Run this section later when you want to evaluate the Han-trained CNN model on the gold dataset.

In [12]:
def load_cnn_artifacts(model_path, tokenizer_path, config_path):
    model = load_model(model_path)

    with open(tokenizer_path, "rb") as f:
        tokenizer = pickle.load(f)

    with open(config_path, "r", encoding="utf-8") as f:
        config = json.load(f)

    return model, tokenizer, config


def build_cnn_features_with_existing_tokenizer(dataframe, tokenizer, max_len):
    pair_text = make_pair_text(dataframe)
    X = pad_sequences(
        tokenizer.texts_to_sequences(pair_text),
        maxlen=max_len,
        padding="post",
        truncating="post"
    )
    return X


def evaluate_saved_cnn_on_gold(
    gold_path,
    model_path,
    tokenizer_path,
    config_path,
    output_prediction_path=None,
    output_summary_path=None
):
    # Load gold dataset
    gold_df = pd.read_csv(gold_path, encoding="utf-8-sig")

    required_cols = ["OrgName_1", "OrgName_2", "Label"]
    missing_cols = [col for col in required_cols if col not in gold_df.columns]
    if missing_cols:
        raise ValueError(f"Missing required columns in gold dataset: {missing_cols}")

    gold_df = gold_df[required_cols].dropna().copy().reset_index(drop=True)
    gold_df["OrgName_1"] = gold_df["OrgName_1"].astype(str)
    gold_df["OrgName_2"] = gold_df["OrgName_2"].astype(str)
    gold_df["Label"] = gold_df["Label"].astype(int)

    # Load saved CNN artifacts
    model, tokenizer, config = load_cnn_artifacts(
        model_path=model_path,
        tokenizer_path=tokenizer_path,
        config_path=config_path
    )

    max_len = config["max_len"]
    batch_size = config.get("batch_size", 32)
    threshold = config.get("threshold", 0.5)
    model_name = config.get("model_name", "CNN")

    # Build input features with the existing tokenizer
    X_gold = build_cnn_features_with_existing_tokenizer(
        gold_df,
        tokenizer,
        max_len=max_len
    )

    y_true = gold_df["Label"].values
    y_prob = model.predict(X_gold, batch_size=batch_size).ravel()
    y_pred, summary_df = evaluate_predictions(
        y_true,
        y_prob,
        model_name=model_name,
        threshold=threshold
    )

    pred_df = gold_df.copy()
    pred_df["Model"] = model_name
    pred_df["TrueLabel"] = y_true
    pred_df["PredictedLabel"] = y_pred
    pred_df["Probability_Match"] = y_prob
    pred_df["Probability_NonMatch"] = 1 - y_prob

    if output_prediction_path is not None:
        pred_df.to_csv(output_prediction_path, index=False, encoding="utf-8-sig")
        print("Saved predictions to:", output_prediction_path)

    if output_summary_path is not None:
        summary_df.to_csv(output_summary_path, index=False, encoding="utf-8-sig")
        print("Saved summary to:", output_summary_path)

    return pred_df, summary_df

## 12. Evaluate the saved Han-trained CNN model on pairwise_gold_dataset.csv

In [13]:
cnn_gold_pred, cnn_gold_summary = evaluate_saved_cnn_on_gold(
    gold_path="pairwise_gold_dataset.csv",
    model_path=MODEL_DIR / "CNN_han_transfer_model.keras",
    tokenizer_path=MODEL_DIR / "CNN_han_transfer_tokenizer.pkl",
    config_path=MODEL_DIR / "CNN_han_transfer_config.json",
    output_prediction_path="cnn_gold_predictions_with_probability.csv",
    output_summary_path="cnn_gold_summary_match_nonmatch.csv"
)

print(cnn_gold_summary)

107/107 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step
              precision    recall  f1-score   support

    NonMatch     0.8670    0.6265    0.7274      2830
       Match     0.2176    0.5194    0.3067       566

    accuracy                         0.6087      3396
   macro avg     0.5423    0.5730    0.5171      3396
weighted avg     0.7588    0.6087    0.6573      3396

Accuracy: 0.6086572438162544
ROC_AUC: 0.599498682715479
Saved predictions to: cnn_gold_predictions_with_probability.csv
Saved summary to: cnn_gold_summary_match_nonmatch.csv
  Model     Class  Precision    Recall        F1  Support  Accuracy   ROC_AUC
0   CNN  NonMatch   0.866993  0.626502  0.727385     2830  0.608657  0.599499
1   CNN     Match   0.217617  0.519435  0.306729      566  0.608657  0.599499
